<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">MODULE 10: METRIC PROCESSING AND SERVICE DELIVERY</div><div style="color:#17212b;font-size:30px;font-weight:750">Define metrics at a declared grain, build a reusable aggregate result, and expose a stable query for downstream dashboards</div><p style="color:#475569;line-height:1.7">Run in order against a dedicated Level 1 course database. Every result is rendered as a table and every write is scoped to this module's objects.</p></div>

## Boundary

This lab never changes the Level 1 source tables. It creates or replaces only objects with the `_l2` suffix.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP TABLE IF EXISTS daily_order_metrics_l2")
lab.execute("""
CREATE TABLE daily_order_metrics_l2 (
    order_date DATE NOT NULL,
    data_source VARCHAR(32) NOT NULL,
    order_count BIGINT SUM NOT NULL DEFAULT "0",
    gross_amount DECIMAL(18,2) SUM NOT NULL DEFAULT "0.00"
)
AGGREGATE KEY(order_date, data_source)
DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.execute("""
INSERT INTO daily_order_metrics_l2
SELECT order_date, data_source, COUNT(*), SUM(order_amount)
FROM orders_imported
GROUP BY order_date, data_source
""")
lab.sql("SELECT * FROM daily_order_metrics_l2 ORDER BY order_date, data_source", title="Serving table at its declared grain")

In [ ]:
lab.sql("""
SELECT order_date,
       SUM(CASE WHEN data_source = 'COURSE_SIMULATION' THEN order_count ELSE 0 END) AS simulated_orders,
       SUM(order_count) AS all_orders,
       SUM(gross_amount) AS gross_amount
FROM daily_order_metrics_l2
GROUP BY order_date
HAVING SUM(order_count) > 0
ORDER BY order_date
""", title="Dashboard metric query")

In [ ]:
lab.sql("""
WITH detail AS (
    SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
    FROM orders_imported
    GROUP BY order_date
), serving AS (
    SELECT order_date, SUM(order_count) AS order_count, SUM(gross_amount) AS gross_amount
    FROM daily_order_metrics_l2
    GROUP BY order_date
)
SELECT d.order_date,
       d.order_count AS detail_orders,
       s.order_count AS serving_orders,
       d.gross_amount AS detail_amount,
       s.gross_amount AS serving_amount
FROM detail AS d
JOIN serving AS s ON s.order_date = d.order_date
ORDER BY d.order_date
""", title="Independent detail-to-serving reconciliation", final=True)

## Takeaway

Compare the result with the business grain stated in the lesson. A successful SQL statement is not by itself evidence that the model, metric, access boundary, or consumer contract is correct.